---
title: "Exercise 9. SNP-Based Heritability"
subtitle: "Post-GWAS Analysis Course"
format:
  html:
    embed-resources: true
    toc: true
    toc-depth: 3
    number-sections: true
execute:
  message: false
  warning: false
jupyter: python
---

# Overview

This notebook estimates SNP-based heritability for ADHD and related traits using LDSC and GWAS summary statistics. You will prepare the inputs, run the heritability analysis, and compare results across traits.

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- explain what SNP-based heritability estimates measure
- prepare GWAS summary statistics for LDSC using the required format and filters
- interpret heritability output across several traits in a comparative way
:::

::: callout-warning
You will need the following input files: GWAS sumstats for ADHD, BMI, educational attainment, age at first birth, and depression. Those are available as input data.
:::

 ## Table of Contents

* [Set up](#section_1)     
* [Prepare the input files for LDSC](#section_2) 
    * [Check headers](#section_2_1)
    * [Munge sumstats](#section_2_2)
* [Run LDSC to estimate the SNP based h2](#section_3)     
* [Analyse and present results](#section_4)

# 1. Set up <a class="anchor" id="section_1"></a>

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

# 2. Pepare the input files for LDSC <a class="anchor" id="section_2"></a>
To estimate the SNP-based heritability (h2-SNP) of the different traits, we are going to use a tool named Linkage Disequilibrium Score Regression (LDSC). (https://github.com/bulik/LDSC)
Several tutorials are available on the github page, including how to estimate h2-SNP: https://github.com/bulik/ldsc/wiki/Heritability-and-Genetic-Correlation

Before running the analysis, we need to harmonise the format of the summary statistics, and restrict them to high quality SNPs found in HapMap3

### 2.1 Inspect headers and sumstats formats <a class="anchor" id="section_2_1"></a>
To harmonize the summary statistics, we will use the munge function from LDSC. This function takes as input the sumstats you want to harmonize, as well as a list of SNP to filter the sumstats. Here we use the HapMap3 SNPs present in the reference file w_hm3.snplist. The function requires that the sumstats have the following columns: SNP (SNP identifier is rs number format), N (sample size), signed statistics column (OR, BETA or Z-score), A1 and A2 (with A1 the effect allele). 
You can use -h to get an overview of the munge function

In [2]:
#Check out the munge function from ldsc

os.system("munge_sumstats -h")

usage: munge_sumstats.py [-h] [--sumstats SUMSTATS] [--N N] [--N-cas N_CAS]
                         [--N-con N_CON] [--out OUT] [--info-min INFO_MIN]
                         [--maf-min MAF_MIN] [--daner] [--daner-n]
                         [--no-alleles] [--merge-alleles MERGE_ALLELES]
                         [--n-min N_MIN] [--chunksize CHUNKSIZE] [--snp SNP]
                         [--N-col N_COL] [--N-cas-col N_CAS_COL]
                         [--N-con-col N_CON_COL] [--a1 A1] [--a2 A2] [--p P]
                         [--frq FRQ] [--signed-sumstats SIGNED_SUMSTATS]
                         [--info INFO] [--info-list INFO_LIST]
                         [--nstudy NSTUDY] [--nstudy-min NSTUDY_MIN]
                         [--ignore IGNORE] [--a1-inc] [--keep-maf]

optional arguments:
  -h, --help            show this help message and exit
  --sumstats SUMSTATS   Input filename.
  --N N                 Sample size If this option is not set, will try to
                        inf

0

As you can see, there are a lot of arguments that we can use to adapt the munging step to the format of the sumstats.
A good practice when using new sumstats is to check the columns and their format as well as the metadata to determine: 
    - What is the genome build? Hg38 or Hg19?
    - Which allele is the effect allele? A1? ALT? EFFECT_ALLELE?
    - Is there a sample size? A number of controls/cases?
    - What is the signed sumstat? OR, BETA, etc
    
You will now inspect the sumstats that we will be using and their metadata to answer these questions and be ready for the munge step. You might have to refer to the original publication or to the method if the information is not available with the data you have here. 

In [3]:
os.makedirs("./output/ldsc", exist_ok=True)

::: callout-note
In input/ldsc open the files with extension .sumstats. You can open them in the file browser of jupyterlab. For each sumstats (ADHD, BMI, educational attainment, age at first birth, depression) answet the following **questions**

- What is the genome build?
- Which allele is the effect allele?
- What is the sample size?
- What is the signed sumstat?
:::

### 2.2 Munge sumstats <a class="anchor" id="section_2_2"></a>
Now that we have all the information we need, we can munge the sumstats. Here is the example on how to do it with the BMI sumstats:

In [4]:
#Example command to run the munge function on the BMI sumstats

command = 'munge_sumstats \
    --sumstats input/ldsc/bmi_yengo_2018.txt \
    --a1 Tested_Allele \
    --out input/ldsc/bmi_yengo_2018_munged \
    --merge-alleles ./reference_data/w_hm3.snplist'

os.system(command)

/opt/ldsc/munge_sumstats.py:419: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True True True ... True True True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  jj[ii] = match


*********************************************************************
* LD Score Regression (LDSC)
* Version 3.0.1
* (C) 2014-2019 Brendan Bulik-Sullivan and Hilary Finucane
* Broad Institute of MIT and Harvard / MIT Department of Mathematics
* GNU General Public License v3
*********************************************************************
Call: 
./munge_sumstats.py \
--sumstats input/ldsc/bmi_yengo_2018.txt \
--out input/ldsc/bmi_yengo_2018_munged \
--merge-alleles ./reference_data/w_hm3.snplist \
--a1 Tested_Allele 

{'SNP': 'SNP', 'Tested_Allele': 'A1', 'Other_Allele': 'A2', 'BETA': 'BETA', 'P': 'P', 'N': 'N'}
Interpreting column names as follows:
SNP:	Variant ID (e.g., rs number)
Tested_Allele:	Allele 1, interpreted as ref allele for signed sumstat.
Other_Allele:	Allele 2, interpreted as non-ref allele for signed sumstat.
BETA:	[linear/logistic] regression coefficient (0 --> no effect; above 0 --> A1 is trait/risk increasing)
P:	p-Value
N:	Sample size

Reading list of SNPs for a

0

In [15]:
#You can now adapt this command and munge the other sumstats 
 #add more lines after --sumstats if needed

#ADHD

command = 'munge_sumstats \
--sumstats ./input/ldsc/ADHD2022_iPSYCH_deCODE_PGC.meta \
--N  \
--out ./output/ldsc/adhd_pgc_2022_munged \
--merge-alleles reference_data/w_hm3.snplist'

os.system(command)

usage: munge_sumstats.py [-h] [--sumstats SUMSTATS] [--N N] [--N-cas N_CAS]
                         [--N-con N_CON] [--out OUT] [--info-min INFO_MIN]
                         [--maf-min MAF_MIN] [--daner] [--daner-n]
                         [--no-alleles] [--merge-alleles MERGE_ALLELES]
                         [--n-min N_MIN] [--chunksize CHUNKSIZE] [--snp SNP]
                         [--N-col N_COL] [--N-cas-col N_CAS_COL]
                         [--N-con-col N_CON_COL] [--a1 A1] [--a2 A2] [--p P]
                         [--frq FRQ] [--signed-sumstats SIGNED_SUMSTATS]
                         [--info INFO] [--info-list INFO_LIST]
                         [--nstudy NSTUDY] [--nstudy-min NSTUDY_MIN]
                         [--ignore IGNORE] [--a1-inc] [--keep-maf]
munge_sumstats.py: error: argument --N-cas: invalid float value: 'Nca'


512

In [ ]:
#You can now adapt this command and munge the other sumstats 
 #add more lines after --sumstats if needed

#ADHD

command = 'munge_sumstats \
--N-cas-col ___ --N-con-col ___ \
--sumstats ./input/ldsc/ADHD2022_iPSYCH_deCODE_PGC.meta \
--out ./output/ldsc/adhd_pgc_2022_munged \
--merge-alleles reference_data/w_hm3.snplist'

os.system(command)

In [ ]:
#Educational attainment

command = 'munge_sumstats \
--sumstats ./input/ldsc/educational_attainment_xia_2025.tsv \
--a1 ___ --a2 ___ \
--snp ___ \
--p ___ \
--N ___ \
--out ./output/ldsc/educational_attainment_xia_2025_munged \
--merge-alleles reference_data/w_hm3.snplist'

os.system(command)

In [ ]:
#Age at first birth 

command = 'munge_sumstats \
--sumstats input/ldsc/year_at_first_birth_mills_2020.tsv \
--a1 ___ --a2 ___ \
--snp ___ \
--p ___ \
--N ___ \
--out output/ldsc/year_at_first_birth_mills_2020_munged \
--merge-alleles reference_data/w_hm3.snplist'

os.system(command)

In [28]:
#Depression (ALREADY written because it has default parameters names)

command = 'munge_sumstats \
--sumstats input/ldsc/depression_pgc_2025.tsv \
--out output/ldsc/depression_pgc_2025_munged \
--merge-alleles reference_data/w_hm3.snplist'

os.system(command)

/opt/ldsc/munge_sumstats.py:419: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True True True ... True True True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  jj[ii] = match


*********************************************************************
* LD Score Regression (LDSC)
* Version 3.0.1
* (C) 2014-2019 Brendan Bulik-Sullivan and Hilary Finucane
* Broad Institute of MIT and Harvard / MIT Department of Mathematics
* GNU General Public License v3
*********************************************************************
Call: 
./munge_sumstats.py \
--sumstats input/ldsc/depression_pgc_2025.tsv \
--out output/ldsc/depression_pgc_2025_munged \
--merge-alleles reference_data/w_hm3.snplist 

{'SNP': 'SNP', 'A1': 'A1', 'A2': 'A2', 'FRQ': 'FRQ', 'INFO': 'INFO', 'BETA': 'BETA', 'P': 'P', 'N': 'N'}
Interpreting column names as follows:
SNP:	Variant ID (e.g., rs number)
A1:	Allele 1, interpreted as ref allele for signed sumstat.
A2:	Allele 2, interpreted as non-ref allele for signed sumstat.
FRQ:	Allele frequency
INFO:	INFO score (imputation quality; higher --> better imputation)
BETA:	[linear/logistic] regression coefficient (0 --> no effect; above 0 --> A1 is trait/risk

0

:::{.callout-note}
A good practice is to check the output log of munge_stats make sure that at least 200,000 SNPs remain in the munged file for each trait
:::

## 3. Run LDSC to estimate the h2-SNP for each trait <a class="anchor" id="section_3"></a>

Now that the sumstats are harmonized, we can calculate the h2-SNP for each trait using ldsc. The function takes as input the munged sumstats, the LD scores for a list of reference SNPs and the list of reference SNPs. 

In [5]:
#Example of command for BMI

command = 'ldsc \
--h2 input/ldsc/bmi_yengo_2018_munged.sumstats.gz \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/bmi_yengo_2018_h2'

os.system(command)

*********************************************************************
* LD Score Regression (LDSC)
* Version 3.0.1
* (C) 2014-2019 Brendan Bulik-Sullivan and Hilary Finucane
* Broad Institute of MIT and Harvard / MIT Department of Mathematics
* GNU General Public License v3
*********************************************************************
Call: 
./ldsc.py \
--out output/ldsc/bmi_yengo_2018_h2 \
--h2 input/ldsc/bmi_yengo_2018_munged.sumstats.gz \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ 

Beginning analysis at Tue Aug 11 13:55:31 2026
Reading summary statistics from input/ldsc/bmi_yengo_2018_munged.sumstats.gz ...
Read summary statistics for 1018612 SNPs.
Reading reference panel LD Score from reference_data/eur_w_ld_chr/[1-22] ... (ldscore_fromlist)
Read reference panel LD Scores for 1290028 SNPs.
Removing partitioned LD Scores with zero variance.
Reading regression weight LD Score from reference_data/eur_w_ld_chr/[1-22] ... (ldscore_fromli

0

In [ ]:
#You can now adapt this command to calculate h2 for the other traits

#ADHD

command = 'ldsc \
--h2 input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/adhd_pgc_2022_h2'

os.system(command)

In [ ]:
#Educational attainment

command = 'ldsc \
--h2 input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/educational_attainment_xia_2025_h2'

os.system(command)

In [ ]:
#Year at first birth

command = 'ldsc \
--h2 input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/year_at_first_birth_mills_2020_h2'

os.system(command)

In [ ]:
#Depression

command = 'ldsc \
--h2 input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/depression_pgc_2025_h2'

os.system(command)

## 4. Analyse and present results <a class="anchor" id="section_4"></a>

You now have calculated the SNP-based heritability for each of the trait. We will extract that information from the log file, and present the results in a table. 

In [35]:
#Create a list with the file paths
path_list = ["output/ldsc/adhd_pgc_2022_h2.log", 
             "output/ldsc/bmi_yengo_2018_h2.log", 
             "output/ldsc/educational_attainment_xia_2025_h2.log", 
             "output/ldsc/year_at_first_birth_mills_2020_h2.log", 
             "output/ldsc/depression_pgc_2025_h2.log"]

#Create a list of trait names
trait_list = ["adhd", "bmi", "educational_attainment", "year_first_birth", "depression"]

#Initiate an empty list 
rows = []

#We iterate over all the files 
for i in range(len(path_list)):

    # Read only the 26th line (index 25) -> line where h2 is
    with open(path_list[i], "r") as f:
        for j, line in enumerate(f):
            if j == 25:
                words = line.strip().split()

                if len(words)>=6:
                    # Extract 5th and 6th "words" (indices 4 and 5) -> corresponds to h2 and se
                    word5 = words[4] 
                    word6 = words[5].strip("()")

                    #Store in rows
                    rows.append({
                        "trait":trait_list[i],
                        "h2_observed": word5,
                        "se": word6
                    })
                break
   
        
# Store in a DataFrame
df = pd.DataFrame(rows)

df

,trait,h2_observed,se
0,adhd,0.1127,0.0052
1,bmi,0.4552,0.0193
2,educational_attainment,0.1082,0.0031
3,year_first_birth,0.0328,0.0012
4,depression,0.0496,0.0016


:::{.callout-note}
**Question.** Based on the result table, are the results in line with what you expect/the literature?
:::